# FastAPI 혼자해보기

# 🚀 FastAPI AI 모델 서빙 실습 - 혼자해보기

---

## 📋 Lab 0: Lifespan

### 🎯 학습 목표

- Lifespan을 활용한 서버 시작/종료 시 리소스 관리
- 전역 저장소를 활용한 모델 공유 방식 이해

### ✅ 혼자해보기

### 과제 1: 다중 모델 로딩 (⭐ 기본)

실습에서는 하나의 모델만 로딩했습니다. 이번에는 **두 개의 가짜 모델**을 로딩해보세요.

**요구사항:**

- `sentiment` 모델 (기존)
- `translator` 모델 (새로 추가)
- 각 모델은 `load_model()` 함수를 통해 로딩
- `/predict/sentiment`와 `/predict/translate` 두 개의 엔드포인트 생성

```python
# 힌트: ml_models 딕셔너리에 여러 모델 저장
ml_models["sentiment"] = load_model()
ml_models["translator"] = load_model()
```

**테스트 방법:**

```bash
curl "http://localhost:8000/predict/sentiment?text=좋아요"
curl "http://localhost:8000/predict/translate?text=안녕하세요"
```

---

### 과제 2: 모델 로딩 시간 측정 (⭐⭐ 심화)

모델 로딩에 걸리는 시간을 측정하고 출력해보세요.

**요구사항:**

- `time` 모듈을 사용해 로딩 시작/종료 시간 측정
- 로딩 완료 후 `"✅ 모델 로딩 완료! (소요시간: X.XX초)"` 형태로 출력
- `/health` 엔드포인트 추가: 모델 로딩 시간 정보 반환

**기대 출력:**

```
====== 모델 로딩중...
✅ 모델 로딩 완료! (소요시간: 3.02초)
```

---

### 과제 3: 환경 설정 분리 (⭐⭐⭐ 도전)

`python-dotenv`를 사용해 모델 로딩 시간을 환경변수로 설정해보세요.

**요구사항:**

- `.env` 파일에 `MODEL_LOAD_TIME=5` 설정
- 해당 값을 읽어서 `time.sleep()`에 적용
- 환경변수가 없으면 기본값 3초 사용

```python
# .env 파일
MODEL_LOAD_TIME=5
```

---

## 📋 Lab 1: 텍스트 요약 API

### 🎯 학습 목표

- HuggingFace Transformers 모델 서빙
- OpenAI API 비동기 호출
- 프롬프트 엔지니어링 기초

### ✅ 혼자해보기

### 과제 1: 요약 길이 옵션 추가 (⭐ 기본)

사용자가 요약 스타일을 선택할 수 있도록 해보세요.

**요구사항:**

- `style` 파라미터 추가: `"short"`, `"medium"`, `"long"`
- 각 스타일별 `min_length`, `max_length` 다르게 설정

| Style | min_length | max_length |
| --- | --- | --- |
| short | 20 | 50 |
| medium | 50 | 150 |
| long | 100 | 300 |

```python
class ArticleRequest(BaseModel):
    text: str
    style: str = "medium"  # 기본값
```

**테스트 방법:**

```bash
curl -X POST "http://localhost:8000/summarize" \
  -H "Content-Type: application/json" \
  -d '{"text": "긴 기사 내용...", "style": "short"}'
```

---

### 과제 2: 감성 분석 API 추가 (⭐⭐ 심화)

요약 API와 함께 **감성 분석 API**를 추가해보세요.

**요구사항:**

- HuggingFace `pipeline("sentiment-analysis")` 사용
- 엔드포인트: `POST /analyze-sentiment`
- 응답: `{"label": "POSITIVE", "score": 0.95}`

```python
# 힌트: Lifespan에서 두 모델 로딩
ml_models["summarizer"] = pipeline("summarization", ...)
ml_models["sentiment"] = pipeline("sentiment-analysis")
```

---

### 과제 3: GPT 키워드 추출기 (⭐⭐⭐ 도전)

OpenAI GPT를 사용해 **키워드 추출 API**를 만들어보세요.

**요구사항:**

- 엔드포인트: `POST /extract-keywords`
- 입력: 긴 텍스트
- 출력: 핵심 키워드 5개 (JSON 배열)
- 시스템 프롬프트로 "데이터 분석가" 역할 부여

**기대 응답:**

```json
{
  "keywords": ["인공지능", "딥러닝", "자연어처리", "트랜스포머", "BERT"]
}
```

---

## 📋 Lab 2: Speech-to-Text (Whisper)

### 🎯 학습 목표

- OpenAI Whisper 모델 로딩 및 서빙
- 파일 업로드 처리 (UploadFile)
- 임시 파일 관리 패턴

### ✅ 혼자해보기

### 과제 1: 지원 형식 검증 (⭐ 기본)

업로드된 파일이 오디오 형식인지 검증해보세요.

**요구사항:**

- 허용 형식: `.mp3`, `.wav`, `.m4a`, `.flac`
- 잘못된 형식 업로드 시 `400 Bad Request` 반환
- 에러 메시지: `"지원하지 않는 파일 형식입니다. (mp3, wav, m4a, flac만 가능)"`

```python
ALLOWED_EXTENSIONS = {".mp3", ".wav", ".m4a", ".flac"}

# 힌트: file.filename에서 확장자 추출
import os
ext = os.path.splitext(file.filename)[1].lower()
```

---

### 과제 2: 타임스탬프 포함 출력 (⭐⭐ 심화)

Whisper의 세그먼트 정보를 활용해 **타임스탬프가 포함된 자막**을 반환해보세요.

**요구사항:**

- 응답에 `segments` 필드 추가
- 각 세그먼트: `{"start": 0.0, "end": 2.5, "text": "안녕하세요"}`

```python
# 힌트: Whisper 결과에는 segments 정보가 포함됨
result = model.transcribe(temp_path)
segments = result.get("segments", [])
```

**기대 응답:**

```json
{
  "text": "전체 텍스트",
  "language": "ko",
  "segments": [
    {"start": 0.0, "end": 2.5, "text": "안녕하세요"},
    {"start": 2.5, "end": 5.0, "text": "오늘 날씨가 좋네요"}
  ]
}
```

---

### 과제 3: 언어 지정 옵션 (⭐⭐⭐ 도전)

사용자가 언어를 미리 지정할 수 있도록 해보세요.

**요구사항:**

- Query Parameter로 `language` 받기 (기본값: 자동 감지)
- Whisper `transcribe()` 메서드의 `language` 옵션 활용
- 지원 언어: `ko`, `en`, `ja`, `zh`

```python
@app.post("/transcribe")
async def transcribe_audio(
    file: UploadFile = File(...),
    language: str = None  # 선택적 파라미터
):
    # language가 지정되면 해당 언어로 강제 인식
    result = model.transcribe(temp_path, language=language)
```

---

## 📋 Lab 3: PyTorch MNIST 모델 서빙

### 🎯 학습 목표

- PyTorch 모델 학습 및 저장
- 학습된 모델(.pth) 로드 및 서빙
- CNN 입력 형태 변환 이해

### ✅ 혼자해보기

### 과제 1: Top-3 예측 반환 (⭐ 기본)

가장 확률이 높은 숫자 3개를 반환해보세요.

**요구사항:**

- 응답에 `top3` 필드 추가
- 각 항목: `{"digit": 7, "confidence": "98.32%"}`

```python
# 힌트: torch.topk() 함수 사용
values, indices = prob.topk(3)
```

**기대 응답:**

```json
{
  "prediction": 7,
  "confidence": "98.32%",
  "top3": [
    {"digit": 7, "confidence": "98.32%"},
    {"digit": 1, "confidence": "1.20%"},
    {"digit": 9, "confidence": "0.35%"}
  ]
}
```

---

### 과제 2: 이미지 파일 업로드 지원 (⭐⭐ 심화)

픽셀 리스트 대신 **이미지 파일을 직접 업로드**할 수 있게 해보세요.

**요구사항:**

- 새 엔드포인트: `POST /predict/image`
- PIL을 사용해 이미지를 28x28 grayscale로 변환
- 기존 추론 로직 재사용

```python
from PIL import Image
import torchvision.transforms as transforms

@app.post("/predict/image")
async def predict_from_image(file: UploadFile = File(...)):
    # 1. 이미지 로드 및 전처리
    image = Image.open(file.file).convert("L")  # Grayscale
    image = image.resize((28, 28))

    # 2. 텐서 변환
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    tensor = transform(image).unsqueeze(0)  # [1, 1, 28, 28]

    # 3. 추론...
```

---

### 과제 3: 배치 추론 API (⭐⭐⭐ 도전)

여러 이미지를 한 번에 추론하는 **배치 API**를 만들어보세요.

**요구사항:**

- 엔드포인트: `POST /predict/batch`
- 입력: 여러 개의 784개 픽셀 리스트
- 한 번의 모델 호출로 모든 이미지 추론

```python
class BatchRequest(BaseModel):
    images: list[list[float]]  # 여러 개의 784개 픽셀 리스트

@app.post("/predict/batch")
async def predict_batch(request: BatchRequest):
    # 배치 텐서 생성: [N, 1, 28, 28]
    batch_tensor = torch.tensor(request.images).view(-1, 1, 28, 28)
    # 한 번에 추론
    with torch.no_grad():
        logits = model(batch_tensor)
    # ...
```

---

## 📋 Lab 4: RAG & LangGraph

### 🎯 학습 목표

- ChromaDB 벡터 저장소 활용
- LCEL 체인 구성
- LangGraph 조건부 라우팅

### ✅ 혼자해보기 (RAG)

### 과제 1: 대화 히스토리 추가 (⭐⭐ 심화)

이전 대화 내용을 기억하는 RAG 시스템을 만들어보세요.

**요구사항:**

- 세션별 대화 히스토리 저장 (메모리 내)
- 프롬프트에 이전 대화 포함
- `session_id`로 세션 구분

```python
# 대화 히스토리 저장소
chat_histories = {}

class QuestionRequest(BaseModel):
    question: str
    session_id: str = "default"

# 프롬프트에 히스토리 포함
template = """Previous conversation:
{history}

Context: {context}
Question: {question}
"""
```

---

### 과제 2: 다중 문서 RAG (⭐⭐⭐ 도전)

여러 문서를 등록하고 검색할 수 있는 RAG를 만들어보세요.

**요구사항:**

- `POST /documents` - 새 문서 추가
- `POST /ask` - 질문 (모든 문서에서 검색)
- 각 문서에 메타데이터 (제목, 카테고리) 포함

```python
class DocumentRequest(BaseModel):
    title: str
    category: str
    content: str

@app.post("/documents")
async def add_document(doc: DocumentRequest):
    new_doc = Document(
        page_content=doc.content,
        metadata={"title": doc.title, "category": doc.category}
    )
    vectorstore.add_documents([new_doc])
    return {"status": "added"}
```

---

### ✅ 혼자해보기 (LangGraph)

### 과제 3: 3단계 라우팅 (⭐⭐ 심화)

기존 2단계(TECHNICAL/CASUAL)에서 **3단계**로 확장해보세요.

**요구사항:**

- 새로운 분류: `CREATIVE` (창작/아이디어 관련)
- `creative_writer` 노드 추가 ("창의적인 작가" 페르소나)
- 조건부 라우팅 수정

```
┌─────────────┐
│  classifier │
└──────┬──────┘
       │
┌──────┴──────┬──────────────┐
│             │              │
▼             ▼              ▼
tech_expert   friendly_bot   creative_writer
```

---

### 과제 4: 대화 루프 에이전트 (⭐⭐⭐ 도전)

사용자가 "종료"라고 할 때까지 대화를 계속하는 에이전트를 만들어보세요.

**요구사항:**

- `/chat/start` - 새 대화 시작
- `/chat/continue` - 대화 이어가기
- "종료", "끝", "bye" 입력 시 대화 종료
- 대화 히스토리 유지

```python
class ChatState(TypedDict):
    messages: list  # 대화 히스토리
    should_end: bool

def check_end(state: ChatState):
    last_msg = state["messages"][-1].lower()
    if any(word in last_msg for word in ["종료", "끝", "bye"]):
        return "end"
    return "continue"
```

---

## 📋 Lab 5: AWS Bedrock

### 🎯 학습 목표

- AWS Bedrock + Claude 모델 호출
- boto3 클라이언트 설정
- OpenAI API와의 차이점 이해

### ✅ 혼자해보기

### 과제 1: 다양한 광고 스타일 (⭐ 기본)

광고 스타일을 선택할 수 있도록 해보세요.

**요구사항:**

- `style` 파라미터 추가: `"formal"`, `"casual"`, `"humorous"`
- 각 스타일별 다른 시스템 프롬프트 적용

```python
class AdRequest(BaseModel):
    product_name: str
    keywords: str
    style: str = "casual"  # formal, casual, humorous

STYLE_PROMPTS = {
    "formal": "당신은 전문 마케터입니다. 격식있고 신뢰감 있는 광고 문구를 작성하세요.",
    "casual": "당신은 친근한 마케터입니다. 편안하고 친근한 광고 문구를 작성하세요.",
    "humorous": "당신은 유머러스한 마케터입니다. 재미있고 기억에 남는 광고 문구를 작성하세요."
}
```

---

### 과제 2: 스트리밍 응답 (⭐⭐⭐ 도전)

응답을 **실시간 스트리밍**으로 반환해보세요.

**요구사항:**

- `invoke_model_with_response_stream()` 메서드 사용
- FastAPI StreamingResponse 활용
- 토큰 단위로 응답 전송

```python
from fastapi.responses import StreamingResponse

@app.post("/generate-ad-stream")
async def generate_ad_stream(request: AdRequest):
    async def generate():
        response = bedrock_client.invoke_model_with_response_stream(...)
        for event in response.get("body"):
            chunk = json.loads(event["chunk"]["bytes"])
            if "delta" in chunk:
                yield chunk["delta"]["text"]

    return StreamingResponse(generate(), media_type="text/plain")

```

---

### 과제 3: 멀티 모델 비교

같은 프롬프트로 **여러 Claude 모델의 결과를 비교**해보세요.

**요구사항:**

- Haiku, Sonnet 두 모델로 동시 호출
- 응답 시간 측정
- 결과 비교 반환

```python
@app.post("/compare-models")
async def compare_models(request: AdRequest):
    import asyncio

    models = [
        "anthropic.claude-3-haiku-20240307-v1:0",
        "anthropic.claude-3-sonnet-20240229-v1:0"
    ]

    results = []
    for model_id in models:
        start = time.time()
        # 호출...
        elapsed = time.time() - start
        results.append({
            "model": model_id,
            "response": result_text,
            "time_seconds": elapsed
        })

    return {"comparisons": results}

```

---

> 🔥 Tip: 막히는 부분이 있다면 실습 코드를 다시 읽어보고, 공식 문서를 참고하세요!
> 
> - [FastAPI 공식 문서](https://fastapi.tiangolo.com/)
> - [HuggingFace Transformers](https://huggingface.co/docs/transformers/)
> - [LangChain 공식 문서](https://python.langchain.com/)
> - [LangGraph 문서](https://langchain-ai.github.io/langgraph/)